In [18]:
from typing import * 

#### 1. Quick Find

In [13]:
class UnionQuickFind:
    def __init__(self,size):
        self.root = [i for i in range(size)]

    # Time complexity - O(1)
    def find(self,x):      
        return self.root[x]

    # Time Complexity - O(N)
    def union(self, x, y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            for i in range(len(self.root)):
                if self.root[i] == root_y:
                    self.root[i] = root_x
    def connected(self, x,y):
        return self.find(x) == self.find(y)


# Test Case
uf = UnionFind(10)
# 1-2-5-6-7 3-8-9 4
uf.union(1, 2)
uf.union(2, 5)
uf.union(5, 6)
uf.union(6, 7)
uf.union(3, 8)
uf.union(8, 9)
print(uf.connected(1, 5))  # true
print(uf.connected(5, 7))  # true
print(uf.connected(4, 9))  # false
# 1-2-5-6-7 3-8-9-4
uf.union(9, 4)
print(uf.connected(4, 9))  # true

True
True
False
True


#### 2. Quick Union

In [14]:
class UnionQuickUnion:
    def __init__(self, size):
        self.root = [i for i in range(size)]

    # Time complexity - O(N)
    def find(self, x):
        while x != self.root[x]:
            x = self.root[x]
        return x

    # Time Complexity = O(N)
    def union(self, x, y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            self.root[y] = root_x

    # Time Complexity - O(N)
    def connected(self,x,y):
        return self.find(x) == self.find(y)

# Test Case
uf = UnionQuickUnion(10)
# 1-2-5-6-7 3-8-9 4
uf.union(1, 2)
uf.union(2, 5)
uf.union(5, 6)
uf.union(6, 7)
uf.union(3, 8)
uf.union(8, 9)
print(uf.connected(1, 5))  # true
print(uf.connected(5, 7))  # true
print(uf.connected(4, 9))  # false
# 1-2-5-6-7 3-8-9-4
uf.union(9, 4)
print(uf.connected(4, 9))  # true

True
True
False
True


#### 3. Union By Rank (Height) - Optimization to Quick Union

# Disjoint Set - Union by Rank

In the previous implementations of **disjoint sets**, we encountered some inefficiencies:

1. In the **quick find** implementation, the **union** operation always takes **O(n)** time.
2. In the **quick union** implementation, as shown in **Figure 6**, it's possible for all the vertices to form a straight line after performing the union operations, which leads to the **worst-case scenario** for the **find** function.

### Is There a Way to Optimize These Implementations?

Yes! The solution to this inefficiency is to implement **union by rank**.

### What is "Rank"?

In this context, **rank** refers to the **height** of the tree representing a vertex. The goal of **union by rank** is to always attach the **shorter tree** (in terms of height) under the **taller tree**, thus preventing the creation of long, unbalanced chains.

### How Does Union by Rank Work?

1. **Rank-based Union**: When performing a union between two vertices, instead of arbitrarily choosing the root node of one of them, we choose the root of the vertex with the **larger rank** (or height).
   
2. **Attaching Trees**: We then attach the root of the **shorter tree** to the root of the **taller tree**. The root of the taller tree becomes the root for both vertices in the union.

3. **Effect**: By always merging the smaller tree into the larger one, we prevent the possibility of the trees becoming too tall and unbalanced. This ensures that the depth of the trees remains **logarithmic**, which makes the **find** function much more efficient.

### Why Does This Work?

The idea is that by maintaining balanced trees, the **find** function becomes faster because we avoid the worst-case scenario where the tree becomes a straight line (which would lead to O(n) time complexity). Instead, the height of the trees grows logarithmically, making the time complexity of both **find** and **union** functions near constant time, **amortized O(α(n))**, where **α(n)** is the inverse Ackermann function.

### Summary

- **Union by rank** improves the performance of the **disjoint set** operations by keeping the trees balanced.
- By always attaching the shorter tree under the taller tree, we prevent the creation of long, unbalanced chains.
- This technique helps to avoid the worst-case scenario and optimizes both the **union** and **find** functions.


In [15]:
class UnionFind:
    # TC = O(N)
    def __init__(self, size):
        self.root = [i for i in range(size)]
        self.rank = [1] * size

    # TC = O(Height) = O(log N)
    def find(self,x):
        while x != self.root[x]:
            x = self.root[x]
        return x

    # TC = O(Height) = O(log N)
    def union(self,x,y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            if self.rank[root_x] > self.rank[root_y]:
                self.root[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.root[root_x] = root_y
            else:
                self.root[y] = root_x
                self.rank[root_x] += 1.   # rank is same, tree size will grow by 1

    # TC = O(Height) = O(log N)
    def connected(self,x,y):
        return self.find(x) == self.find(y)
        
# Test Case
uf = UnionFind(10)
# 1-2-5-6-7 3-8-9 4
uf.union(1, 2)
uf.union(2, 5)
uf.union(5, 6)
uf.union(6, 7)
uf.union(3, 8)
uf.union(8, 9)
print(uf.connected(1, 5))  # true
print(uf.connected(5, 7))  # true
print(uf.connected(4, 9))  # false
# 1-2-5-6-7 3-8-9-4
uf.union(9, 4)
print(uf.connected(4, 9))  # true              

True
True
False
True


# Path Compression Optimization - Disjoint Sets

In the previous implementation of the **disjoint set**, when we use the `find` function to locate the root node, we traverse the parent nodes **sequentially** until we reach the root node. However, if we perform the same search again for the same element, we repeat the same steps, which is inefficient.

### Can We Optimize This Process?

Yes! We can improve this process using an optimization technique called **path compression**.

### Path Compression

After finding the root node, we update the parent node of all traversed elements to point directly to the **root node**. This means that the next time we search for the root node of the same element, we won’t need to traverse all the parent nodes again. Instead, we only need to traverse a very small number of nodes, making the process **much more efficient**.

### How Does It Work?

1. **Find the root node**: We perform the usual process of finding the root node for a given element.
2. **Update the parent nodes**: While traversing up the tree to find the root, we update the parent pointer of each element to point directly to the root node.
3. **Efficiency boost**: The next time we call the `find` function for any of the elements, we can directly access the root node with minimal traversal, significantly speeding up the operation.

### Using Recursion

The process of updating the parent nodes to point to the root is done using **recursion**. This ensures that the path compression is applied to all elements along the traversal path.

### Summary

- **Path compression** optimizes the `find` function by reducing the depth of the tree.
- It significantly improves performance by reducing redundant operations when searching for the same element multiple times.
- The **recursion** mechanism ensures that all traversed elements are efficiently updated to directly point to the root node.



In [16]:
class UnionFind:
    # TC = O(N)
    def __init__(self, size):
        self.root = [i for i in range(size)]
        self.rank = [1] * size

    # TC = O(α(N)) - α refers to the Inverse Ackermann function. 
    # In practice, we assume it's a constant. In other words, O(α(N)) is regarded as O(1) on average.
    def find(self,x):
        if x == self.root[x]:
            return x
        self.root[x] = self.find(self.root[x])
        return self.root[x]

    # TC = O(α(N)) - α refers to the Inverse Ackermann function. 
    # In practice, we assume it's a constant. In other words, O(α(N)) is regarded as O(1) on average.
    def union(self,x,y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            if self.rank[root_x] > self.rank[root_y]:
                self.root[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.root[root_x] = root_y
            else:
                self.root[y] = root_x
                self.rank[root_x] += 1.   # rank is same, tree size will grow by 1

    # TC = O(α(N)) - α refers to the Inverse Ackermann function. 
    # In practice, we assume it's a constant. In other words, O(α(N)) is regarded as O(1) on average.
    def connected(self,x,y):
        return self.find(x) == self.find(y)
        
# Test Case
uf = UnionFind(10)
# 1-2-5-6-7 3-8-9 4
uf.union(1, 2)
uf.union(2, 5)
uf.union(5, 6)
uf.union(6, 7)
uf.union(3, 8)
uf.union(8, 9)
print(uf.connected(1, 5))  # true
print(uf.connected(5, 7))  # true
print(uf.connected(4, 9))  # false
# 1-2-5-6-7 3-8-9-4
uf.union(9, 4)
print(uf.connected(4, 9))  # true              

True
True
False
True


# Disjoint Set Data Structure

The **disjoint set** data structure is used to represent a collection of disjoint sets. The key idea is that all **connected vertices** (either directly or indirectly) share the same **parent node** or **root node**. This allows us to efficiently check if two vertices are connected by simply checking if they share the same root.

### Key Functions

There are two main operations in the disjoint set:

1. **`find` function**:
   - The `find` function is used to locate the **root node** of a given vertex. It helps identify which set the vertex belongs to.

2. **`union` function**:
   - The `union` function is used to connect two previously **unconnected vertices** by making them share the same root node, essentially merging their sets.

3. **`connected` function**:
   - The `connected` function checks whether two vertices are in the same set. It works by checking if both vertices have the same root node.

### Summary
- The **find** and **union** functions are essential for efficiently managing and querying connectivity in the disjoint set data structure.
- These functions are widely used in many problems, such as determining the connected components in a graph.



### 1. Number of Provinces

In [19]:
class UnionFind:
    def __init__(self, size):
        self.root = [i for i in range(size)]
        self.rank = [1] * size
    
    def find(self, x):
        if x == self.root[x]:
            return x
        self.root[x] = self.find(self.root[x])
        return self.root[x]

    def union(self, x,y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            if self.rank[root_x] > self.rank[root_y]:
                self.root[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.root[root_x] = root_y
            else:
                self.root[root_y] = root_x
                self.rank[root_x] += 1
    
    def connected(self, x,y):
        return self.find(x) == self.find(y)

    def total_roots(self):
        roots = 0
        for i in range(len(self.root)):
            if i == self.root[i]:
                roots += 1
        return roots
    
class Solution:
    def findCircleNum(self, isConnected: List[List[int]]) -> int:
        n = len(isConnected)
        uf = UnionFind(n)
        for i in range(n):
            for j in range(i+1,n):
                if isConnected[i][j]:
                    uf.union(i,j)
        return uf.total_roots()

#### 2. Valid Graph Tree

#### Problem Description
You are given a graph of `n` nodes labeled from `0` to `n - 1`. You are also given a list of edges where `edges[i] = [a_i, b_i]` indicates that there is an **undirected edge** between nodes `a_i` and `b_i`.

**Return `true` if the edges of the given graph form a valid tree, and `false` otherwise.**

##### Definition of a Tree:
A **tree** is an undirected graph in which:
- There are no cycles.
- All nodes are connected, meaning there is exactly one path between any two nodes.

##### Constraints:
- `1 <= n <= 2000`
- `0 <= edges.length <= 5000`
- `edges[i].length == 2`
- `0 <= a_i, b_i < n`
- `a_i != b_i`
- There are no self-loops or repeated edges.

In [21]:
class UnionFind:
    def __init__(self, size):
        self.root = [i for i in range(size)]
        self.rank = [1] * size
    
    def find(self, x):
        if x == self.root[x]:
            return x
        self.root[x] = self.find(self.root[x])
        return self.root[x]

    def union(self, x,y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            if self.rank[root_x] > self.rank[root_y]:
                self.root[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.root[root_x] = root_y
            else:
                self.root[root_y] = root_x
                self.rank[root_x] += 1
            return True
        else:
            return False
    
    def connected(self, x,y):
        return self.find(x) == self.find(y)

    def total_roots(self):
        roots = 0
        for i in range(len(self.root)):
            if i == self.root[i]:
                roots += 1
        return roots

class Solution:
    def validTree(self, n: int, edges: List[List[int]]) -> bool:
        # Any graph without a cycle is a tree
        uf = UnionFind(n)

        for u,v in edges:
            if uf.union(u, v) == False:
                return False 
        
        if uf.total_roots() > 1:
            return False
        return True

#### 3. The Earliest Moment When Everyone Become Friends

#### Problem Description
There are `n` people in a social group labeled from `0` to `n - 1`.  
You are given an array `logs` where `logs[i] = [timestamp_i, x_i, y_i]` indicates that `x_i` and `y_i` become friends at time `timestamp_i`.

Friendship is **symmetric**. That means:
- If person `a` is friends with person `b`, then `b` is also friends with `a`.

A person `a` is **acquainted** with person `b` if:
- `a` is directly friends with `b`, **or**
- `a` is friends with someone who is acquainted with `b` (i.e., indirectly connected).

Your task is to **return the earliest timestamp** at which **every person** became acquainted with **every other person**.  
If there is **no such time**, return `-1`.

---

#### Input
- Integer `n` – total number of people, labeled `0` to `n-1`
- List of logs: `logs[i] = [timestamp_i, x_i, y_i]`

---

#### Output
- Integer – the **earliest timestamp** when everyone is acquainted, or `-1` if it never happens

In [22]:
class UnionFind:
    def __init__(self, size):
        self.root = [i for i in range(size)]
        self.rank = [1] * size
    
    def find(self, x):
        if x == self.root[x]:
            return x
        self.root[x] = self.find(self.root[x])
        return self.root[x]

    def union(self, x,y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            if self.rank[root_x] > self.rank[root_y]:
                self.root[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.root[root_x] = root_y
            else:
                self.root[root_y] = root_x
                self.rank[root_x] += 1
    
    def connected(self, x,y):
        return self.find(x) == self.find(y)

    def total_roots(self):
        roots = 0
        for i in range(len(self.root)):
            if i == self.root[i]:
                roots += 1
        return roots

class Solution:
    def earliestAcq(self, logs: List[List[int]], n: int) -> int:
        logs.sort(key = lambda x: x[0])
        uf = UnionFind(n)
        for t,u,v in logs:
            uf.union(u,v)
            if uf.total_roots() == 1:
                return t
        return -1

#### Smallest String With Swaps

#### Problem Description
You are given:
- A string `s`
- A list of index pairs `pairs`, where `pairs[i] = [a, b]` indicates that you can **swap** the characters at indices `a` and `b` (0-indexed)

You can perform **any number of swaps** using the pairs in `pairs`.

**Task:**  
Return the **lexicographically smallest** string that `s` can be changed to after applying the allowed swaps.

---

#### Input
- A string `s`
- A list of integer pairs `pairs[i] = [a, b]`

---

#### Output
- A string – the **lexicographically smallest** string possible after all swaps

In [23]:
class UnionFind:
    def __init__(self, size):
        self.root = [i for i in range(size)]
        self.rank = [1] * size
    
    def find(self, x):
        if x == self.root[x]:
            return x
        self.root[x] = self.find(self.root[x])
        return self.root[x]

    def union(self, x,y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            if self.rank[root_x] > self.rank[root_y]:
                self.root[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.root[root_x] = root_y
            else:
                self.root[root_y] = root_x
                self.rank[root_x] += 1
    
    def connected(self, x,y):
        return self.find(x) == self.find(y)

    def total_roots(self):
        roots = 0
        for i in range(len(self.root)):
            if i == self.root[i]:
                roots += 1
        return roots

    def get_roots(self):
        root_nodes = []
        for i in range(len(self.root)):
            if i == self.root[i]:
                root_nodes.append(i)
        return root_nodes

class Solution:
    def smallestStringWithSwaps(self, s: str, pairs: List[List[int]]) -> str:
        n = len(s)
        uf = UnionFind(n)
        for x,y in pairs:
            uf.union(x, y)
        
        root_nodes = {x: deque() for x in uf.get_roots()}
        ans = [''] * n
        for i in range(n):
            root_nodes[uf.find(i)].append(i)

        for root in root_nodes:
            chars = [s[i] for i in root_nodes[root]]
            chars.sort()
            i = 0
            for idx in root_nodes[root]:
                ans[idx] = chars[i]
                i += 1
        
        return ''.join(ans)
